# LightGBM Hyperparameter Tuning with Optuna

This notebook focus on optimizing the LightGBM classifier using Bayesian search for multiclass churn prediction.

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
import joblib
import os
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

C:\Users\MONISH\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_data():
    logger.info("Loading split data...")
    X_train = pd.read_csv('../../data/04_features/X_train.csv')
    y_train = pd.read_csv('../../data/04_features/y_train.csv').squeeze().astype(int)
    X_test = pd.read_csv('../../data/04_features/X_test.csv')
    y_test = pd.read_csv('../../data/04_features/y_test.csv').squeeze().astype(int)
    return X_train, X_test, y_train, y_test

X_train_full, X_test, y_train_full, y_test = load_data()

# Create a validation set for Optuna and Early Stopping
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

logger.info(f"Train set: {X_train.shape}, Val set: {X_val.shape}, Test set: {X_test.shape}")

2026-04-14 18:37:22,838 - INFO - Loading split data...


2026-04-14 18:37:22,911 - INFO - Train set: (4096, 20), Val set: (1025, 20), Test set: (1281, 20)


In [3]:
def objective(trial):
    param = {
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_logloss',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'random_state': 42,
        'class_weight': 'balanced',
        'n_estimators': trial.suggest_int('n_estimators', 200, 800),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'num_leaves': trial.suggest_int('num_leaves', 10, 50),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
    }

    model = lgb.LGBMClassifier(**param)
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='multi_logloss',
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False)
        ]
    )

    preds = model.predict(X_val)
    score = f1_score(y_val, preds, average='macro')
    return score

In [4]:
logger.info("Starting optimization...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

logger.info("Optimization finished.")
logger.info(f"Best trial score (Macro F1): {study.best_value}")
logger.info(f"Best parameters: {study.best_params}")

2026-04-14 18:37:22,944 - INFO - Starting optimization...


[I 2026-04-14 18:37:22,948] A new study created in memory with name: no-name-e2293c05-a2db-4ed6-8432-bbb42a45299c


[I 2026-04-14 18:37:28,632] Trial 0 finished with value: 0.7585953747846913 and parameters: {'n_estimators': 424, 'learning_rate': 0.04343869376540922, 'max_depth': 8, 'num_leaves': 16, 'min_child_samples': 25, 'subsample': 0.9900455638021189, 'colsample_bytree': 0.7420012179127382, 'reg_alpha': 2.719960731823906, 'reg_lambda': 0.9641521487072574}. Best is trial 0 with value: 0.7585953747846913.


[I 2026-04-14 18:37:29,355] Trial 1 finished with value: 0.7418414974015163 and parameters: {'n_estimators': 622, 'learning_rate': 0.0452016135835397, 'max_depth': 7, 'num_leaves': 43, 'min_child_samples': 34, 'subsample': 0.6174971629520727, 'colsample_bytree': 0.7930767548905624, 'reg_alpha': 1.5243066351910146, 'reg_lambda': 3.7474503276010633}. Best is trial 0 with value: 0.7585953747846913.


[I 2026-04-14 18:37:30,026] Trial 2 finished with value: 0.7598241773455547 and parameters: {'n_estimators': 776, 'learning_rate': 0.06484676198612777, 'max_depth': 5, 'num_leaves': 30, 'min_child_samples': 20, 'subsample': 0.859734108935319, 'colsample_bytree': 0.6953893539848139, 'reg_alpha': 1.0897259912174628, 'reg_lambda': 4.161019901031262}. Best is trial 2 with value: 0.7598241773455547.


...

[I 2026-04-14 18:37:48,901] Trial 28 finished with value: 0.760823282372244 and parameters: {'n_estimators': 570, 'learning_rate': 0.02544183461250254, 'max_depth': 7, 'num_leaves': 29, 'min_child_samples': 93, 'subsample': 0.7701612805280801, 'colsample_bytree': 0.9687090268875982, 'reg_alpha': 3.0018806981733075, 'reg_lambda': 2.399485376775851}. Best is trial 3 with value: 0.7692281922909777.


[I 2026-04-14 18:37:49,722] Trial 29 finished with value: 0.7443287449602204 and parameters: {'n_estimators': 468, 'learning_rate': 0.03982384613297617, 'max_depth': 6, 'num_leaves': 11, 'min_child_samples': 46, 'subsample': 0.9507129968041043, 'colsample_bytree': 0.7135916625926556, 'reg_alpha': 2.648050344578734, 'reg_lambda': 0.9060397713943816}. Best is trial 3 with value: 0.7692281922909777.


2026-04-14 18:37:49,722 - INFO - Optimization finished.


2026-04-14 18:37:49,726 - INFO - Best trial score (Macro F1): 0.7692281922909777


2026-04-14 18:37:49,728 - INFO - Best parameters: {'n_estimators': 778, 'learning_rate': 0.046315319457752385, 'max_depth': 7, 'num_leaves': 50, 'min_child_samples': 68, 'subsample': 0.8803604498658864, 'colsample_bytree': 0.6830030978521533, 'reg_alpha': 0.9164340100180196, 'reg_lambda': 3.3084876311488}


In [5]:
def train_final_model(best_params, X_train, y_train, X_validation, y_validation):
    logger.info("Training final model with best parameters...")
    
    final_params = {
        'objective': 'multiclass',
        'num_class': 3,
        'random_state': 42,
        'class_weight': 'balanced',
        **best_params
    }
    
    model = lgb.LGBMClassifier(**final_params)
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_validation, y_validation)],
        eval_metric='multi_logloss',
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=50)
        ]
    )
    
    return model

# Retrain on full training data, using test data for early stopping validation
final_model = train_final_model(study.best_params, X_train_full, y_train_full, X_test, y_test)

2026-04-14 18:37:49,755 - INFO - Training final model with best parameters...


Training until validation scores don't improve for 50 rounds
[50]	valid_0's multi_logloss: 0.395596


[100]	valid_0's multi_logloss: 0.366889
[150]	valid_0's multi_logloss: 0.364918


[200]	valid_0's multi_logloss: 0.365424
Early stopping, best iteration is:
[169]	valid_0's multi_logloss: 0.364437


In [6]:
def evaluate_model(model, X_test, y_test):
    logger.info("Evaluating final model...")
    preds = model.predict(X_test)
    
    print(f"\nAccuracy: {accuracy_score(y_test, preds)}")
    print("\nClassification Report:\n", classification_report(y_test, preds))
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, preds))
    
    logger.info("Saving model...")
    os.makedirs('../../models', exist_ok=True)
    joblib.dump(model, '../../models/lightgbm_churn_model.pkl')
    logger.info("Model saved to ../../models/lightgbm_churn_model.pkl")

evaluate_model(final_model, X_test, y_test)

2026-04-14 18:37:50,607 - INFO - Evaluating final model...



Accuracy: 0.8157689305230289


2026-04-14 18:37:50,665 - INFO - Saving model...


2026-04-14 18:37:50,716 - INFO - Model saved to ../../models/lightgbm_churn_model.pkl



Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.63      0.70       427
           1       0.48      0.70      0.57       214
           2       1.00      0.98      0.99       640

    accuracy                           0.82      1281
   macro avg       0.76      0.77      0.75      1281
weighted avg       0.84      0.82      0.82      1281


Confusion Matrix:
 [[271 156   0]
 [ 65 149   0]
 [  7   8 625]]
